# TechCompenso job scraping

Scrape job postings from techcompenso.com using Selenium.

## 1. Imports

In [1]:
# import the libraries we need
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import sqlite3
import re
import time

## 2. Launch browser

Start a headless Chrome instance used to render the site.

In [2]:
# start a headless Chrome browser
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
driver = webdriver.Chrome(options=options)

## 3. Scrape all pages

Loop over the 15 listing pages and collect every job card.

In [3]:
# collect raw card text and link from every page
all_cards = []
seen_urls = set()
for page in range(1, 16):
    # open the jobs page for this page number
    driver.get(f"https://techcompenso.com/offerte-di-lavoro?page={page}")
    time.sleep(3)
    # find every job card on the page
    cards = driver.find_elements(By.CSS_SELECTOR, "div[data-spotlight]")
    new_count = 0
    for card in cards:
        # the detail link is an overlay anchor inside the card
        link = ""
        try:
            link = card.find_element(By.CSS_SELECTOR, "a[href*='/offerte-di-lavoro/']").get_attribute("href")
        except Exception:
            pass
        # skip cards we already collected
        if link in seen_urls:
            continue
        seen_urls.add(link)
        # store the full card text plus its link
        all_cards.append({"text": card.text, "url": link})
        new_count += 1
    print(f"page {page}: {len(cards)} cards, {new_count} new")
print("total unique cards:", len(all_cards))

page 1: 20 cards, 20 new
page 2: 20 cards, 10 new
page 3: 20 cards, 10 new
page 4: 20 cards, 10 new
page 5: 20 cards, 10 new
page 6: 20 cards, 10 new
page 7: 20 cards, 10 new
page 8: 20 cards, 10 new
page 9: 20 cards, 10 new
page 10: 20 cards, 10 new
page 11: 20 cards, 10 new
page 12: 20 cards, 10 new
page 13: 20 cards, 10 new
page 14: 20 cards, 10 new
page 15: 18 cards, 8 new
total unique cards: 158


## 4. Parse card text into columns

Extract title, company, location, work mode, salary range and tags from each card.

In [4]:
# turn raw card text into structured fields
def parse(card):
    # split the card text into non-empty lines
    lines = []
    for l in card["text"].splitlines():
        l = l.strip()
        if l:
            lines.append(l)
    # drop the optional "Hiring Partner" badge
    if lines and lines[0] == "Hiring Partner":
        lines = lines[1:]
    # first line is the job title, second is the company
    title = None
    company = None
    if len(lines) > 0:
        title = lines[0]
    if len(lines) > 1:
        company = lines[1]
    # find salary, work mode and location among the remaining lines
    salary = None
    work_mode = None
    location = None
    for ln in lines[2:]:
        if "€" in ln:
            salary = ln
        elif ln in {"Full-Remote", "Ibrido", "In sede", "Remote", "Hybrid"}:
            work_mode = ln
        elif location is None and not any(ch.isdigit() for ch in ln) and len(ln) < 40:
            location = ln
    # anything left over becomes tags
    tags = []
    for ln in lines[2:]:
        if ln not in (salary, work_mode, location):
            tags.append(ln)
    # split the salary range into two numeric columns
    salary_min = None
    salary_max = None
    if salary:
        nums = []
        for n in re.findall(r"[\d\.]+", salary):
            nums.append(float(n.replace(".", "").replace(",", ".")))
        if "fino a" in salary.lower():
            if nums:
                salary_max = nums[0]
        else:
            if len(nums) > 0:
                salary_min = nums[0]
            if len(nums) > 1:
                salary_max = nums[1]
    return {"title": title, "company": company, "location": location,
            "work_mode": work_mode, "salary_min": salary_min, "salary_max": salary_max,
            "tags": ", ".join(tags), "url": card["url"]}

# parse every card
rows = []
for c in all_cards:
    rows.append(parse(c))

# deduplicate by url
seen = set()
unique = []
for r in rows:
    if r["url"] not in seen:
        seen.add(r["url"])
        unique.append(r)

# build a DataFrame from the unique postings
df = pd.DataFrame(unique)
df

,title,company,location,work_mode,salary_min,salary_max,tags,url
0,iOS Engineer,Welyk x Laura,Milano,Ibrido,50000.0,70000.0,,https://techcompenso.com/offerte-di-lavoro/ios...
1,Senior Frontend Developer,Welyk x Laura,Milano,Ibrido,50000.0,70000.0,,https://techcompenso.com/offerte-di-lavoro/sen...
2,Android Developer,Welyk x Laura,Milano,Ibrido,50000.0,70000.0,,https://techcompenso.com/offerte-di-lavoro/and...
3,Senior Backend Developer,Welyk x Laura,Milano,Ibrido,50000.0,70000.0,,https://techcompenso.com/offerte-di-lavoro/sen...
4,Backend Engineer,Welyk x Tundr,NaN,Full-Remote,40000.0,50000.0,,https://techcompenso.com/offerte-di-lavoro/bac...
...,...,...,...,...,...,...,...,...
153,Product Manager (Risks) Europe,JustMarkets,RAL non disponibile,Full-Remote,NaN,NaN,"FuffAnnuncio Score 3.2, Dettagli, Middle, Prod...",https://techcompenso.com/offerte-di-lavoro/pro...
154,Senior Sales Development Representative - EMEA,Sprinto,RAL non disponibile,Full-Remote,NaN,NaN,"FuffAnnuncio Score 3.3, Dettagli, Senior, Sale...",https://techcompenso.com/offerte-di-lavoro/sen...
155,Brand Design Lead Europe,JustMarkets,RAL non disponibile,Full-Remote,NaN,NaN,"FuffAnnuncio Score 3.3, Dettagli, Senior, Desi...",https://techcompenso.com/offerte-di-lavoro/bra...
156,Senior Data Engineer,Nansen,RAL non disponibile,Full-Remote,NaN,NaN,"FuffAnnuncio Score 3.3, Dettagli, Senior, Data...",https://techcompenso.com/offerte-di-lavoro/sen...


## 5. Persist to SQLite

Write the parsed postings to the techcompenso table in job_postings.db.

In [5]:
# save the postings into job_postings.db under the techcompenso table
db_path = "job_postings.db"
conn = sqlite3.connect(db_path)
if not df.empty:
    df = df.set_index("url")
    df.to_sql("techcompenso", conn, if_exists="replace", index=True, index_label="url")
    conn.commit()
    print(f"Saved {len(df)} postings")
conn.close()
driver.quit()

Saved 158 postings
